In [1]:
import subprocess, os, socket

def port_open(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(1)
        return s.connect_ex(('127.0.0.1', port)) == 0

if not port_open(6000):
    subprocess.Popen(
        ['python3.11', '-m', 'mlflow', 'server', '-p', '6000', '-h', '0.0.0.0',
         '--backend-store-uri', '/srv/mlruns', '--workers', '1'],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
        env={**os.environ, 'PYTHONPATH': '/srv/python_packages:' + os.environ.get('PYTHONPATH', '')}
    )
    import time; time.sleep(3)
    print("MLflow iniciado en http://localhost:6000")
else:
    print("MLflow ya estaba corriendo en puerto 6000")

import mlflow
mlflow.set_tracking_uri("http://localhost:6000")
mlflow.set_experiment("latam2cloud")
print(f"MLflow OK — {mlflow.get_tracking_uri()}  |  experimento: latam2cloud")

MLflow iniciado en http://localhost:6000
MLflow OK — http://localhost:6000  |  experimento: latam2cloud


In [2]:
import sys
if '/srv/python_packages' not in sys.path:
    sys.path.insert(0, '/srv/python_packages')

import lightgbm
print(f"lightgbm {lightgbm.__version__} OK — path: {lightgbm.__file__}")

lightgbm 4.7.0 OK — path: /srv/python_packages/lightgbm/__init__.py


In [3]:
import sys
if '/srv/python_packages' not in sys.path:
    sys.path.insert(0, '/srv/python_packages')

import os, sys, time, gc, warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
import pyarrow.parquet as pq
from sklearn.metrics import (cohen_kappa_score, f1_score,
                             balanced_accuracy_score, confusion_matrix)
from sklearn.utils.class_weight import compute_sample_weight
warnings.filterwarnings("ignore")

SAVE_DIR   = '/srv/work/models'
IN_FEAT    = '/srv/work/te_features_v6.parquet'
ROUTE_KEYS = ['source_agent', 'cloud_provider', 'region']
ID_COLS    = ROUTE_KEYS + ['window']
SPLIT_COL  = 'split'
TARGET_COL = 'label_ventana'
HORIZON    = 2

T1_THRESHOLDS = np.round(np.arange(0.05, 0.51, 0.01), 2)

LGB_PARAMS_V13 = {
    'objective': 'multiclass', 'num_class': 4, 'metric': 'multi_logloss',
    'num_leaves': 63, 'learning_rate': 0.03, 'n_estimators': 800,
    'max_depth': -1, 'min_child_samples': 30,
    'subsample': 0.8, 'colsample_bytree': 0.8,
    'reg_alpha': 0.1, 'reg_lambda': 1.0,
    'n_jobs': 8, 'random_state': 42, 'verbosity': -1,
}
SW_POWER      = 0.5
QWK_FLOOR     = 0.65
RECALTA_FLOOR = 0.55

print(f'LGB v13: num_leaves={LGB_PARAMS_V13["num_leaves"]}, '
      f'lr={LGB_PARAMS_V13["learning_rate"]}, n_est={LGB_PARAMS_V13["n_estimators"]}')
print(f'SW_POWER={SW_POWER} | T1: {len(T1_THRESHOLDS)} umbrales [{T1_THRESHOLDS[0]}–{T1_THRESHOLDS[-1]}]')

LGB v13: num_leaves=63, lr=0.03, n_est=800
SW_POWER=0.5 | T1: 46 umbrales [0.05–0.5]


In [4]:
import pyarrow.parquet as pq

print('Leyendo schema...')
schema    = pq.read_schema(IN_FEAT)
all_cols  = [f.name for f in schema]

exclude   = set(ID_COLS + [TARGET_COL, SPLIT_COL])
FEAT_COLS = [c for c in all_cols if c not in exclude]
keep_cols = ID_COLS + [TARGET_COL, SPLIT_COL] + FEAT_COLS
print(f'  Features: {len(FEAT_COLS)} | Columnas a leer: {len(keep_cols)}')

print('Leyendo train...')
df_train = pq.read_table(IN_FEAT,
    columns=keep_cols,
    filters=[('split','=','train')]
).to_pandas()
df_train['y'] = df_train.groupby(ROUTE_KEYS)[TARGET_COL].shift(-HORIZON)
df_train = df_train.dropna(subset=['y']).copy()
df_train['y'] = df_train['y'].astype(int)
X_train = df_train[FEAT_COLS].values.astype(np.float32)
y_train = df_train['y'].values
del df_train; gc.collect()
print(f'  X_train: {X_train.shape} | Alta: {(y_train==3).sum():,}')

print('Leyendo val...')
df_val = pq.read_table(IN_FEAT,
    columns=keep_cols,
    filters=[('split','=','val')]
).to_pandas()
df_val['y'] = df_val.groupby(ROUTE_KEYS)[TARGET_COL].shift(-HORIZON)
df_val = df_val.dropna(subset=['y']).copy()
df_val['y'] = df_val['y'].astype(int)
X_val = df_val[FEAT_COLS].values.astype(np.float32)
y_val = df_val['y'].values
del df_val; gc.collect()
print(f'  X_val:   {X_val.shape}   | Alta: {(y_val==3).sum():,}')

sw_full          = compute_sample_weight("balanced", y_train)
sample_weight_t2 = sw_full ** SW_POWER
print(f'  sample_weight: [{sample_weight_t2.min():.4f}, {sample_weight_t2.max():.4f}]')
print('Celda 2 OK')

Leyendo schema...
  Features: 103 | Columnas a leer: 109
Leyendo train...
  X_train: (5347524, 103) | Alta: 106,527
Leyendo val...
  X_val:   (1138777, 103)   | Alta: 21,503
  sample_weight: [0.5379, 3.5426]
Celda 2 OK


In [5]:

def evaluar(y_true, y_pred_proba, nombre, floor_qwk=QWK_FLOOR, floor_rec=RECALTA_FLOOR):
    y_pred = np.argmax(y_pred_proba, axis=1)
    qwk    = cohen_kappa_score(y_true, y_pred, weights="quadratic")
    cm     = confusion_matrix(y_true, y_pred, labels=[0,1,2,3])
    rec_alta = cm[3,3] / cm[3,:].sum() if cm[3,:].sum() > 0 else 0.0
    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    bal_acc  = balanced_accuracy_score(y_true, y_pred)
    ok_qwk   = qwk >= floor_qwk
    ok_rec   = rec_alta >= floor_rec
    cand     = "✅ SÍ" if (ok_qwk and ok_rec) else "❌ NO"
    print('-'*80)
    print(f'  {nombre}')
    print('-'*80)
    print(f'  QWK      = {qwk:.4f}  {"✅" if ok_qwk else "❌"}  (floor≥{floor_qwk})')
    print(f'  RecAlta  = {rec_alta:.4f}  {"✅" if ok_rec else "❌"}  (floor≥{floor_rec})')
    print(f'  MacroF1  = {macro_f1:.4f}')
    print(f'  BalAcc   = {bal_acc:.4f}')
    print(f'  CANDIDATO: {cand}')
    return qwk, rec_alta, macro_f1, bal_acc

print('evaluar() definida.')


evaluar() definida.


In [6]:

print('T1: Cargando ens_v10_val_proba.npy...')
proba_ens_v10 = np.load(f'{SAVE_DIR}/ens_v10_val_proba.npy')
print(f'  ens_v10: {proba_ens_v10.shape}')

y_pred_base = np.argmax(proba_ens_v10, axis=1)
qwk_base  = cohen_kappa_score(y_val, y_pred_base, weights="quadratic")
cm_base   = confusion_matrix(y_val, y_pred_base, labels=[0,1,2,3])
rec_base  = cm_base[3,3] / cm_base[3,:].sum()
print(f'  Baseline argmax: QWK={qwk_base:.4f}, RecAlta={rec_base:.4f}')

t1_results = []
for T in T1_THRESHOLDS:
    y_pred = np.argmax(proba_ens_v10, axis=1).copy()
    alta_mask = proba_ens_v10[:, 3] > T
    y_pred[alta_mask] = 3
    qwk     = cohen_kappa_score(y_val, y_pred, weights="quadratic")
    cm      = confusion_matrix(y_val, y_pred, labels=[0,1,2,3])
    rec_alta = cm[3,3] / cm[3,:].sum() if cm[3,:].sum() > 0 else 0.0
    is_cand = (qwk >= QWK_FLOOR) and (rec_alta >= RECALTA_FLOOR)
    t1_results.append((T, qwk, rec_alta, is_cand))

print('\n  {:>6}  {:>8}  {:>8}  Cand'.format('T', 'QWK', 'RecAlta'))
print('  ' + '-'*40)
for T, qwk, rec, cand in t1_results:
    mark = '✅' if cand else ''
    print(f'  {T:>6.2f}  {qwk:>8.4f}  {rec:>8.4f}   {mark}')

candidates_t1 = [(T, qwk, rec) for T, qwk, rec, c in t1_results if c]
print(f'\nT1: {len(candidates_t1)} combinaciones candidatas encontradas')


T1: Cargando ens_v10_val_proba.npy...
  ens_v10: (1138777, 4)
  Baseline argmax: QWK=0.6231, RecAlta=0.5254

       T       QWK   RecAlta  Cand
  ----------------------------------------
    0.05    0.2310    0.9401   
    0.06    0.2755    0.9284   
    0.07    0.3179    0.9157   
    0.08    0.3574    0.9022   
    0.09    0.3938    0.8899   
    0.10    0.4264    0.8768   
    0.11    0.4551    0.8624   
    0.12    0.4785    0.8477   
    0.13    0.4970    0.8344   
    0.14    0.5128    0.8212   
    0.15    0.5264    0.8073   
    0.16    0.5370    0.7940   
    0.17    0.5466    0.7819   
    0.18    0.5544    0.7705   
    0.19    0.5618    0.7591   
    0.20    0.5679    0.7490   
    0.21    0.5736    0.7373   
    0.22    0.5786    0.7252   
    0.23    0.5836    0.7139   
    0.24    0.5883    0.7034   
    0.25    0.5925    0.6915   
    0.26    0.5968    0.6792   
    0.27    0.6009    0.6673   
    0.28    0.6051    0.6542   
    0.29    0.6082    0.6403   
    0.30    0

In [7]:

if candidates_t1:
    T_best, qwk_best_t1, rec_best_t1 = max(candidates_t1, key=lambda x: x[1])
    print(f'T1 MEJOR UMBRAL: T={T_best} → QWK={qwk_best_t1:.4f}, RecAlta={rec_best_t1:.4f} ✅')
    y_pred_t1 = np.argmax(proba_ens_v10, axis=1).copy()
    y_pred_t1[proba_ens_v10[:, 3] > T_best] = 3
    proba_t1 = proba_ens_v10.copy()
    qwk_t1, rec_t1, f1_t1, ba_t1 = evaluar(y_val, proba_ens_v10,
        f'Ens v10 + T1 threshold={T_best} (post-hoc)', QWK_FLOOR, RECALTA_FLOOR)
    qwk_t1   = cohen_kappa_score(y_val, y_pred_t1, weights="quadratic")
    cm_t1    = confusion_matrix(y_val, y_pred_t1, labels=[0,1,2,3])
    rec_t1   = cm_t1[3,3] / cm_t1[3,:].sum()
    print(f'  (con y_pred override) QWK={qwk_t1:.4f}, RecAlta={rec_t1:.4f}')
    np.save(f'{SAVE_DIR}/ens_v10_t1_val_proba.npy', proba_ens_v10)
    print(f'  ens_v10_t1_val_proba.npy guardado (threshold T={T_best} aplicar en inferencia)')
else:
    T_best = None; qwk_t1 = None; rec_t1 = None
    print('T1: ningún umbral genera candidato.')
    best_qwk_row = max(t1_results, key=lambda x: x[1])
    best_rec_row = max(t1_results, key=lambda x: x[2])
    print(f'  Mejor QWK:    T={best_qwk_row[0]} QWK={best_qwk_row[1]:.4f} RecAlta={best_qwk_row[2]:.4f}')
    print(f'  Mejor RecAlta: T={best_rec_row[0]} QWK={best_rec_row[1]:.4f} RecAlta={best_rec_row[2]:.4f}')


T1: ningún umbral genera candidato.
  Mejor QWK:    T=0.49 QWK=0.6231 RecAlta=0.5254
  Mejor RecAlta: T=0.05 QWK=0.2310 RecAlta=0.9401


In [8]:

t_start = time.time()
print(f'T2: LGB v13 training — num_leaves={LGB_PARAMS_V13["num_leaves"]}, '
      f'lr={LGB_PARAMS_V13["learning_rate"]}, n_est={LGB_PARAMS_V13["n_estimators"]}')
print(f'  X_train: {X_train.shape}  |  SW_POWER={SW_POWER}')

lgb_v13 = lgb.LGBMClassifier(**LGB_PARAMS_V13)
lgb_v13.fit(
    X_train, y_train,
    sample_weight=sample_weight_t2,
    eval_set=[(X_val, y_val)],
    callbacks=[
        lgb.log_evaluation(100),
        lgb.early_stopping(stopping_rounds=50, verbose=False),
    ],
)

t_elapsed = time.time() - t_start
print(f'  Entrenado en {t_elapsed/60:.1f} min')

proba_lgb_v13 = lgb_v13.predict_proba(X_val)
print(f'  Predicción val: proba_lgb_v13 {proba_lgb_v13.shape}')
np.save(f'{SAVE_DIR}/lgb_v13_val_proba.npy', proba_lgb_v13)
lgb_v13.booster_.save_model(f'{SAVE_DIR}/lgb_v13_model.txt')
print(f'  Guardado: lgb_v13_val_proba.npy + lgb_v13_model.txt')

qwk_lgb13, rec_lgb13, f1_lgb13, ba_lgb13 = evaluar(
    y_val, proba_lgb_v13, 'LGB v13 (num_leaves=63, lr=0.03, 800 est, sqrt-balanced)')

del X_train; gc.collect()
print('  X_train liberado')

del sample_weight_t2; gc.collect()


T2: LGB v13 training — num_leaves=63, lr=0.03, n_est=800
  X_train: (5347524, 103)  |  SW_POWER=0.5
[100]	valid_0's multi_logloss: 0.238711
[200]	valid_0's multi_logloss: 0.228994
[300]	valid_0's multi_logloss: 0.223643
[400]	valid_0's multi_logloss: 0.220324
[500]	valid_0's multi_logloss: 0.218597
[600]	valid_0's multi_logloss: 0.217672
[700]	valid_0's multi_logloss: 0.216536
[800]	valid_0's multi_logloss: 0.215438
  Entrenado en 20.3 min
  Predicción val: proba_lgb_v13 (1138777, 4)
  Guardado: lgb_v13_val_proba.npy + lgb_v13_model.txt
--------------------------------------------------------------------------------
  LGB v13 (num_leaves=63, lr=0.03, 800 est, sqrt-balanced)
--------------------------------------------------------------------------------
  QWK      = 0.6022  ❌  (floor≥0.65)
  RecAlta  = 0.5320  ❌  (floor≥0.55)
  MacroF1  = 0.6513
  BalAcc   = 0.6781
  CANDIDATO: ❌ NO
  X_train liberado


0

In [9]:

print('T2: Cargando checkpoint RF v10...')
proba_rf_v10 = np.load(f'{SAVE_DIR}/rf_v10_val_proba.npy')
print(f'  proba_rf_v10: {proba_rf_v10.shape}')

best_t2 = None
print(f'\nGrid search RF_v10 × LGB_v13 — paso 0.05:')
print('  {:>6}  {:>6}  {:>8}  {:>8}  Cand'.format('w_RF','w_LGB','QWK','RecAlta'))
print('  ' + '-'*50)
for w_rf in np.arange(0.0, 1.01, 0.05):
    w_lgb = round(1.0 - w_rf, 2)
    proba_ens = w_rf * proba_rf_v10 + w_lgb * proba_lgb_v13
    y_pred = np.argmax(proba_ens, axis=1)
    qwk    = cohen_kappa_score(y_val, y_pred, weights="quadratic")
    cm     = confusion_matrix(y_val, y_pred, labels=[0,1,2,3])
    rec    = cm[3,3] / cm[3,:].sum() if cm[3,:].sum() > 0 else 0.0
    is_cand = (qwk >= QWK_FLOOR) and (rec >= RECALTA_FLOOR)
    mark = '✅' if is_cand else ''
    print(f'  {w_rf:>6.2f}  {w_lgb:>6.2f}  {qwk:>8.4f}  {rec:>8.4f}   {mark}')
    if is_cand:
        if best_t2 is None or qwk > best_t2[2]:
            best_t2 = (w_rf, w_lgb, qwk, rec, proba_ens.copy())

if best_t2:
    w_rf_b, w_lgb_b, qwk_t2, rec_t2, proba_ens_t2 = best_t2
    print(f'\nT2 CANDIDATO: RF×{w_rf_b}+LGB×{w_lgb_b} → QWK={qwk_t2:.4f}, RecAlta={rec_t2:.4f} ✅')
    np.save(f'{SAVE_DIR}/ens_v13_val_proba.npy', proba_ens_t2)
    print(f'  ens_v13_val_proba.npy guardado')
    evaluar(y_val, proba_ens_t2,
            f'Ensemble v13 óptimo (RF×{w_rf_b}+LGB×{w_lgb_b})')
else:
    qwk_t2 = None; rec_t2 = None; w_rf_b = None; w_lgb_b = None
    best_by_qwk = None; best_qwk_val = -1
    for w_rf in np.arange(0.0, 1.01, 0.05):
        w_lgb = round(1.0 - w_rf, 2)
        proba_ens = w_rf * proba_rf_v10 + w_lgb * proba_lgb_v13
        y_pred = np.argmax(proba_ens, axis=1)
        qwk = cohen_kappa_score(y_val, y_pred, weights="quadratic")
        if qwk > best_qwk_val:
            best_qwk_val = qwk
            cm = confusion_matrix(y_val, y_pred, labels=[0,1,2,3])
            rec = cm[3,3] / cm[3,:].sum() if cm[3,:].sum() > 0 else 0.0
            best_by_qwk = (w_rf, w_lgb, qwk, rec, proba_ens.copy())
    w_rf_b, w_lgb_b, qwk_t2, rec_t2, proba_ens_t2 = best_by_qwk
    np.save(f'{SAVE_DIR}/ens_v13_val_proba.npy', proba_ens_t2)
    print(f'\nT2: sin candidato — mejor QWK: RF×{w_rf_b}+LGB×{w_lgb_b}'
          f' QWK={qwk_t2:.4f} RecAlta={rec_t2:.4f}')
    evaluar(y_val, proba_ens_t2,
            f'Ensemble v13 óptimo (RF×{w_rf_b}+LGB×{w_lgb_b})')


T2: Cargando checkpoint RF v10...
  proba_rf_v10: (1138777, 4)

Grid search RF_v10 × LGB_v13 — paso 0.05:
    w_RF   w_LGB       QWK   RecAlta  Cand
  --------------------------------------------------
    0.00    1.00    0.6022    0.5320   
    0.05    0.95    0.6041    0.5299   
    0.10    0.90    0.6061    0.5276   
    0.15    0.85    0.6078    0.5251   
    0.20    0.80    0.6095    0.5227   
    0.25    0.75    0.6110    0.5204   
    0.30    0.70    0.6124    0.5194   
    0.35    0.65    0.6141    0.5184   
    0.40    0.60    0.6156    0.5174   
    0.45    0.55    0.6169    0.5170   
    0.50    0.50    0.6177    0.5147   
    0.55    0.45    0.6188    0.5134   
    0.60    0.40    0.6197    0.5124   
    0.65    0.35    0.6203    0.5113   
    0.70    0.30    0.6208    0.5099   
    0.75    0.25    0.6212    0.5096   
    0.80    0.20    0.6219    0.5096   
    0.85    0.15    0.6224    0.5092   
    0.90    0.10    0.6222    0.5087   
    0.95    0.05    0.6217    0.5073  

In [10]:

print('='*80)
print('  TABLA COMPARATIVA v9 → v10 → v11 → v12 → v13 (T1+T2) (validación)')
print('='*80)
print('  {:^55}  {:>6}  {:>8}  Cand'.format('Modelo','QWK','RecAlta'))
print('  ' + '-'*80)

rows = [
    ('LGB v13 (T2 individual)', qwk_lgb13, rec_lgb13),
    (f'Ensemble v13 T2 óptimo (RF×{w_rf_b}+LGB×{w_lgb_b})', qwk_t2, rec_t2),
]

if T_best is not None:
    rows.insert(0, (f'Ens v10 + T1 threshold={T_best}', qwk_t1, rec_t1))

for nombre, qwk, rec in rows:
    if qwk is None: continue
    cand = '✅' if (qwk >= QWK_FLOOR and rec >= RECALTA_FLOOR) else '❌'
    print(f'  {nombre:<55}  {qwk:>6.4f}  {rec:>8.4f}  {cand}')

print('  ' + '-'*80)
ref_rows = [
    ('-- REF v9 (83d, v5) -- Ens D38',       0.6529, 0.6509, 'ok'),
    ('-- REF v10 (162d, v6) -- Ens optimo',   0.6227, 0.5237, '--'),
    ('-- REF v11 (83d, v6/P1) -- Ens optimo', 0.6231, 0.5254, '--'),
    ('-- REF v12 (162d/P2) -- XGB Ens opt',   0.5410, 0.7048, '--'),
]
for nm, q, r, mk in ref_rows:
    print(f'  {nm:<55}  {q:>6.4f}  {r:>8.4f}  {mk}')
print()
print(f'  Umbral candidato: QWK≥{QWK_FLOOR} AND RecAlta≥{RECALTA_FLOOR}  (Benchmarking Criterios Éxito v2.0)')

t1_cand = T_best is not None and qwk_t1 >= QWK_FLOOR and rec_t1 >= RECALTA_FLOOR
t2_cand = qwk_t2 is not None and qwk_t2 >= QWK_FLOOR and rec_t2 >= RECALTA_FLOOR
any_cand = t1_cand or t2_cand

if any_cand:
    print('  CANDIDATO FINAL v13: ✅ — ver LOG-48')
else:
    print('  CANDIDATO FINAL v13: ❌ ninguno')
    print('  → Candidato OE2 definitivo: v9 Ensemble D38 (LOG-25)')
    print('  → Proceder con 06 evaluacion test set v7.ipynb (one-shot)')


  TABLA COMPARATIVA v9 → v10 → v11 → v12 → v13 (T1+T2) (validación)
                          Modelo                              QWK   RecAlta  Cand
  --------------------------------------------------------------------------------
  LGB v13 (T2 individual)                                  0.6022    0.5320  ❌
  Ensemble v13 T2 óptimo (RF×0.8500000000000001+LGB×0.15)  0.6224    0.5092  ❌
  --------------------------------------------------------------------------------
  -- REF v9 (83d, v5) -- Ens D38                           0.6529    0.6509  ok
  -- REF v10 (162d, v6) -- Ens optimo                      0.6227    0.5237  --
  -- REF v11 (83d, v6/P1) -- Ens optimo                    0.6231    0.5254  --
  -- REF v12 (162d/P2) -- XGB Ens opt                      0.5410    0.7048  --

  Umbral candidato: QWK≥0.65 AND RecAlta≥0.55  (Benchmarking Criterios Éxito v2.0)
  CANDIDATO FINAL v13: ❌ ninguno
  → Candidato OE2 definitivo: v9 Ensemble D38 (LOG-25)
  → Proceder con 06 evaluacion tes

In [11]:

import mlflow

with mlflow.start_run(run_name='lgb_v13') as run:
    mlflow.log_params({**LGB_PARAMS_V13, 'sw_power': SW_POWER})
    mlflow.log_metrics({'qwk_val': qwk_lgb13, 'rec_alta_val': rec_lgb13,
                         'macro_f1_val': f1_lgb13, 'bal_acc_val': ba_lgb13})
    mlflow.log_artifact(f'{SAVE_DIR}/lgb_v13_model.txt')
    print(f'MLflow run lgb_v13 registrado — {run.info.run_id}')

with mlflow.start_run(run_name='ensemble_v13') as run:
    mlflow.log_params({'w_rf': w_rf_b, 'w_lgb': w_lgb_b,
                        'rf_source': 'rf_v10_ckpt', 'lgb_source': 'lgb_v13',
                        't1_threshold': T_best if T_best else 'N/A',
                        't1_candidato': t1_cand, 't2_candidato': t2_cand})
    mlflow.log_metrics({'qwk_t2_val': qwk_t2 or 0, 'rec_t2_val': rec_t2 or 0})
    if T_best:
        mlflow.log_metrics({'qwk_t1_val': qwk_t1, 'rec_t1_val': rec_t1,
                             't1_threshold': T_best})
    print(f'MLflow run ensemble_v13 registrado — {run.info.run_id}')

if any_cand:
    print('\n⭐ CANDIDATO v13 CONGELADO — proceder con 06 evaluacion test set v7.ipynb')
else:
    print('\n→ Sin candidato en v13. Candidato OE2 final: v9 Ensemble D38 (LOG-25).')
    print('  Próximo paso: 06 evaluacion test set v7.ipynb (one-shot con v9 Ens D38)')


MLflow run lgb_v13 registrado — 456268f83d2c4e088f40530704f2ff59
🏃 View run lgb_v13 at: http://localhost:6000/#/experiments/681062908553326840/runs/456268f83d2c4e088f40530704f2ff59
🧪 View experiment at: http://localhost:6000/#/experiments/681062908553326840
MLflow run ensemble_v13 registrado — 0bd42e17eab544d3a243b196f69c21b1
🏃 View run ensemble_v13 at: http://localhost:6000/#/experiments/681062908553326840/runs/0bd42e17eab544d3a243b196f69c21b1
🧪 View experiment at: http://localhost:6000/#/experiments/681062908553326840

→ Sin candidato en v13. Candidato OE2 final: v9 Ensemble D38 (LOG-25).
  Próximo paso: 06 evaluacion test set v7.ipynb (one-shot con v9 Ens D38)
